# LeetCode 310 — Minimum Height Trees bằng Python

Link đề: https://leetcode.com/problems/minimum-height-trees/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Topological Sort

Skeleton LeetCode:

```python
class Solution:
    def findMinHeightTrees(self, n: int, edges: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **cây** (đồ thị vô hướng liên thông, không chu trình) gồm `n` nút đánh số `0..n−1`; chọn nút nào làm **gốc** thì ta được một cây có gốc với một **chiều cao** (số cạnh trên đường dài nhất từ gốc xuống lá). Cần trả về **mọi** nút gốc cho chiều cao **nhỏ nhất** (các "Minimum Height Trees", MHT).
2. Thấy các hướng đi: **thử mọi gốc bằng BFS** (`O(n²)`) → **bóc lá dần (topological sort trên lá)** → **tìm đường kính rồi lấy tâm** → **quy hoạch động đổi gốc (rerooting)** → **dùng DFS tìm đường kính theo hai lượt**.
3. Hiểu **vì sao đáp án chỉ có 1 hoặc 2 nút** và là **tâm của đường kính** của cây.
4. Biết cách **kiểm thử bài có nhiều đáp án đúng**: so sánh **tập đã sắp xếp** (hoặc kiểm tra tính chất), không so thứ tự.
5. Tránh lỗi kinh điển: quên trường hợp `n = 1` (không có cạnh nên không có lá), dừng bóc lá sai điều kiện (`remaining > 2`), chỉ trả một tâm khi đường kính có số cạnh lẻ, và chọn "nút bậc cao nhất" (sai).

## 2. Đề bài & ví dụ

Một cây là đồ thị vô hướng liên thông và không có chu trình. Cho một cây gồm `n` nút gán nhãn `0..n−1` và mảng `edges` có `n − 1` cạnh, `edges[i] = [aᵢ, bᵢ]` là một cạnh vô hướng giữa `aᵢ` và `bᵢ`. Bạn có thể chọn **bất kỳ nút nào** làm gốc; khi đó **chiều cao** của cây là số cạnh trên đường đi dài nhất từ gốc xuống một lá. Trong mọi cây có gốc có thể có, những cây có **chiều cao nhỏ nhất** là các **cây chiều cao tối thiểu (MHT)**. Trả về **danh sách nhãn gốc của mọi MHT** (thứ tự bất kỳ).

**Ví dụ 1:** `n = 4`, `edges = [[1,0],[1,2],[1,3]]` → `[1]`. Chọn gốc `1` cho chiều cao `1`; gốc khác cho chiều cao `2`.

**Ví dụ 2:** `n = 6`, `edges = [[3,0],[3,1],[3,2],[3,4],[5,4]]` → `[3,4]`.

**Ràng buộc:** `1 <= n <= 2 * 10⁴`; `edges.length == n − 1`; `0 <= aᵢ, bᵢ < n`; `aᵢ != bᵢ`; mọi cặp `(aᵢ, bᵢ)` khác nhau; đồ thị đã cho đảm bảo là một cây.

In [1]:
from typing import List
from collections import deque, defaultdict
import random

examples = [
    (4, [[1, 0], [1, 2], [1, 3]], [1]),
    (6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]], [3, 4]),
]
for n, e, ans in examples:
    print("n =", n, "| edges =", e, "-> kỳ vọng (tập)", ans)

n = 4 | edges = [[1, 0], [1, 2], [1, 3]] -> kỳ vọng (tập) [1]
n = 6 | edges = [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]] -> kỳ vọng (tập) [3, 4]


## 3. Kiến thức nền

### 3.1. Chiều cao của cây theo gốc (BFS theo tầng)

Từ một gốc `r`, chạy BFS: khoảng cách lớn nhất từ `r` đến mọi nút là **chiều cao** của cây có gốc `r` (đây cũng là "độ lệch tâm" - eccentricity - của `r`).

In [2]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    return adj


def height_from(adj, root):
    n = len(adj)
    dist = [-1] * n
    dist[root] = 0
    q = deque([root])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1
                q.append(v)
    return max(dist)


adj = build_adj(4, [[1, 0], [1, 2], [1, 3]])
print("chiều cao theo từng gốc:", [height_from(adj, r) for r in range(4)])

chiều cao theo từng gốc: [2, 1, 2, 2]


### 3.2. Lá, bậc và đường kính

- **Lá**: nút có đúng một cạnh (bậc 1).
- **Đường kính** của cây: số cạnh của đường đi **dài nhất** giữa hai nút bất kỳ. Hai đầu của đường kính luôn là lá.
- **Tâm** của đường kính: nút (hoặc hai nút kề nhau) ở **giữa** đường đi đó. Nếu đường kính có `D` cạnh thì chiều cao nhỏ nhất bằng `⌈D/2⌉`.

Hai bước BFS tìm đường kính: BFS từ nút bất kỳ để lấy nút xa nhất `u`; BFS từ `u` để lấy nút xa nhất `v`; khi đó `u – v` là một đường kính (tính chất kinh điển của cây).

In [3]:
def bfs_far(adj, src):
    n = len(adj)
    dist = [-1] * n
    par = [-1] * n
    dist[src] = 0
    q = deque([src])
    last = src
    while q:
        u = q.popleft()
        last = u
        for v in adj[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1
                par[v] = u
                q.append(v)
    return last, dist, par


adj = build_adj(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]])
u, _, _ = bfs_far(adj, 0)
v, dist, par = bfs_far(adj, u)
path = [v]
while par[path[-1]] != -1:
    path.append(par[path[-1]])
print("một đường kính:", path, "| số cạnh D =", dist[v])

một đường kính: [2, 3, 4, 5] | số cạnh D = 3


### 3.3. Bóc lá (leaf peeling) = topological sort trên lá

Bỏ **mọi lá** cùng lúc, rồi các nút vừa trở thành lá lại bị bỏ ở lượt sau, ... Mỗi lượt bỏ đi một "lớp vỏ". Khi còn **1 hoặc 2 nút** thì dừng: đó chính là tâm. Cách làm giống **sắp xếp topo (Kahn)** trên đồ thị vô hướng, với điều kiện "bậc còn lại bằng 1".

In [4]:
n, edges = 6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]]
adj = build_adj(n, edges)
deg = [len(a) for a in adj]
layer = [i for i in range(n) if deg[i] == 1]
print("lớp lá đầu tiên:", layer, "| bậc:", deg)

lớp lá đầu tiên: [0, 1, 2, 5] | bậc: [1, 1, 1, 4, 2, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung cây như cái cây thật: muốn **chiều cao nhỏ nhất**, đặt gốc ở **chính giữa**. Ví dụ đường đi `0 − 1 − 2 − 3 − 4` (5 nút): gốc 0 cao 4, gốc 1 cao 3, gốc 2 cao 2 (tốt nhất), gốc 3 cao 3, gốc 4 cao 4. Đường đi `0 − 1 − 2 − 3` (4 nút): gốc `1` cao 2 và gốc `2` cao 2: **hai tâm**.

Cách "bóc lá": ở đường đi 5 nút, lượt 1 bỏ `0` và `4`, lượt 2 bỏ `1` và `3`, còn `2`. Ở đường đi 4 nút, lượt 1 bỏ `0` và `3`, còn `1` và `2` (đã ≤ 2 nút: dừng).

**Vì sao chỉ có tối đa 2 tâm:** tâm nằm giữa đường kính. Nếu có 3 nút cùng chiều cao tối thiểu thì đường kính sẽ ngắn hơn: mâu thuẫn (xem mục 10).

In [5]:
for n, edges in [(5, [[0, 1], [1, 2], [2, 3], [3, 4]]), (4, [[0, 1], [1, 2], [2, 3]])]:
    adj = build_adj(n, edges)
    hs = [height_from(adj, r) for r in range(n)]
    print("n =", n, "| chiều cao theo gốc:", hs, "| MHT:", [i for i in range(n) if hs[i] == min(hs)])

n = 5 | chiều cao theo gốc: [4, 3, 2, 3, 4] | MHT: [2]
n = 4 | chiều cao theo gốc: [3, 2, 2, 3] | MHT: [1, 2]


## 5. Cách 1 — Brute Force / baseline: thử từng nút làm gốc

**Ý tưởng.** Với mỗi nút `r`, BFS tính chiều cao; giữ các nút có chiều cao nhỏ nhất. `n` lần BFS, mỗi lần `O(n)`: **`O(n²)`**.

In [6]:
def find_mht_brute(n: int, edges: List[List[int]]) -> List[int]:
    adj = build_adj(n, edges)
    heights = [height_from(adj, r) for r in range(n)]
    best = min(heights)
    return [r for r in range(n) if heights[r] == best]


for n, e, ans in examples:
    assert sorted(find_mht_brute(n, e)) == sorted(ans)
print("PASS ví dụ (brute force)")

PASS ví dụ (brute force)


**Độ phức tạp (Cách 1).** Time `O(n²)` (với `n = 2·10⁴` là `4·10⁸`: quá chậm); Space `O(n)`. Dùng làm **oracle**.

**Nút thắt.** Ta tính lại gần như cùng một thông tin cho mỗi gốc. MHT thực chất chỉ liên quan đến **chỗ giữa** của cây.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát:** chiều cao khi chọn gốc `r` bằng **khoảng cách xa nhất** từ `r` đến một lá; và nút xa nhất từ `r` luôn là một đầu của một đường kính. Vì thế chiều cao nhỏ nhất đạt được khi `r` ở **giữa một đường kính**: chiều cao `= ⌈D/2⌉`.

- Nếu `D` **chẵn** (số cạnh chẵn, số nút lẻ): đúng **một** tâm.
- Nếu `D` **lẻ**: **hai** tâm kề nhau.

Bóc lá chính là **gỡ dần hai đầu của mọi đường kính**; còn lại 1 hoặc 2 nút là tâm. Hai cách "bóc lá" và "tìm đường kính" thực chất là cùng một ý.

In [7]:
# Kiểm chứng quan sát bằng cách vét cạn: số MHT chỉ là 1 hoặc 2, hai nút này kề nhau, chiều cao nhỏ nhất = ceil(D / 2)
def random_tree(n, rnd):
    perm = list(range(n))
    rnd.shuffle(perm)
    edges = []
    for i in range(1, n):
        j = rnd.randrange(i) if rnd.random() < 0.7 else i - 1
        a, b = perm[i], perm[j]
        edges.append([a, b] if rnd.random() < 0.5 else [b, a])
    rnd.shuffle(edges)
    return edges


rnd = random.Random(310)
for _ in range(500):
    n = rnd.randint(1, 30)
    edges = random_tree(n, rnd)
    adj = build_adj(n, edges)
    hs = [height_from(adj, r) for r in range(n)]
    best = min(hs)
    centers = [r for r in range(n) if hs[r] == best]
    u, _, _ = bfs_far(adj, 0)
    v, dist, _ = bfs_far(adj, u)
    D = dist[v]
    assert len(centers) in (1, 2) and best == (D + 1) // 2
    if len(centers) == 2:
        assert centers[1] in adj[centers[0]]
    assert len(centers) == 1 + D % 2
print("PASS: 500 cây ngẫu nhiên có 1 hoặc 2 tâm kề nhau, chiều cao nhỏ nhất = ceil(D / 2)")

PASS: 500 cây ngẫu nhiên có 1 hoặc 2 tâm kề nhau, chiều cao nhỏ nhất = ceil(D / 2)


## 7. Các cách tiếp theo

### Cách 2 — Bóc lá theo từng lớp (topological sort, cách tham khảo)

**Giải thích.** Xây danh sách kề và mảng bậc. Hàng đợi ban đầu chứa mọi lá. Trong khi còn **hơn 2 nút**: trừ đi số lá hiện tại, với mỗi lá giảm bậc của hàng xóm; hàng xóm có bậc còn `1` thì vào lớp kế. Dừng khi còn `≤ 2` nút: lớp lá hiện hành chính là các tâm. Trường hợp `n ≤ 2` trả thẳng mọi nút (`n = 1` không có lá).

In [8]:
def find_mht_peel(n: int, edges: List[List[int]]) -> List[int]:
    if n <= 2:
        return list(range(n))
    adj = build_adj(n, edges)
    deg = [len(a) for a in adj]
    leaves = [i for i in range(n) if deg[i] == 1]
    remaining = n
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for nb in adj[leaf]:
                deg[nb] -= 1
                if deg[nb] == 1:
                    nxt.append(nb)
        leaves = nxt
    return leaves


for n, e, ans in examples:
    assert sorted(find_mht_peel(n, e)) == sorted(ans)
print("PASS ví dụ (bóc lá)")

PASS ví dụ (bóc lá)


**Độ phức tạp Cách 2:** Time `O(n)` (mỗi nút bị bóc đúng một lần, mỗi cạnh duyệt hai lần); Space `O(n)`.

### Cách 3 — Hai lượt BFS tìm đường kính rồi lấy tâm

**Giải thích.** BFS từ nút `0` được nút xa nhất `u`; BFS từ `u` được nút xa nhất `v` và mảng cha. Truy vết từ `v` về `u` được đường đi `p[0..D]` (`D` cạnh). Tâm: `p[D // 2]`, và nếu `D` lẻ thêm `p[D // 2 + 1]`.

In [9]:
def find_mht_diameter(n: int, edges: List[List[int]]) -> List[int]:
    if n == 1:
        return [0]
    adj = build_adj(n, edges)
    u, _, _ = bfs_far(adj, 0)
    v, dist, par = bfs_far(adj, u)
    path = [v]
    while par[path[-1]] != -1:
        path.append(par[path[-1]])
    D = len(path) - 1
    if D % 2 == 0:
        return [path[D // 2]]
    return [path[D // 2], path[D // 2 + 1]]


for n, e, ans in examples:
    assert sorted(find_mht_diameter(n, e)) == sorted(ans)
print("PASS ví dụ (đường kính)")

PASS ví dụ (đường kính)


**Độ phức tạp Cách 3:** Time `O(n)`; Space `O(n)`.

### Cách 4 — Quy hoạch động đổi gốc (rerooting), tính chiều cao cho mọi gốc

**Giải thích.** Chọn gốc `0`. `down[v]` = chiều cao cây con của `v`. `up[v]` = độ dài đường dài nhất từ `v` đi **lên** cha rồi xuống nhánh khác. Với con `c` của `v`: `up[c] = 1 + max(up[v], giá_trị_xuống_tốt_nhất_của_các_anh_em_c)`, trong đó ta giữ hai giá trị `down + 1` lớn nhất của các con của `v`. Chiều cao khi gốc là `v` bằng `max(down[v], up[v])`. Làm bằng vòng lặp (không đệ quy) theo thứ tự BFS.

In [10]:
def find_mht_reroot(n: int, edges: List[List[int]]) -> List[int]:
    if n == 1:
        return [0]
    adj = build_adj(n, edges)
    parent = [-1] * n
    order = [0]
    for u in order:                                      # BFS order (order vừa duyệt vừa mở rộng)
        for v in adj[u]:
            if v != parent[u]:
                parent[v] = u
                order.append(v)
    down = [0] * n
    for u in reversed(order):
        p = parent[u]
        if p >= 0:
            down[p] = max(down[p], down[u] + 1)
    best1 = [0] * n                                      # hai giá trị down[c] + 1 lớn nhất của các con
    best2 = [0] * n
    for u in order:
        for c in adj[u]:
            if c == parent[u]:
                continue
            val = down[c] + 1
            if val > best1[u]:
                best1[u], best2[u] = val, best1[u]
            elif val > best2[u]:
                best2[u] = val
    up = [0] * n
    for u in order:
        for c in adj[u]:
            if c == parent[u]:
                continue
            sibling = best2[u] if down[c] + 1 == best1[u] else best1[u]
            up[c] = 1 + max(up[u], sibling)
    heights = [max(down[v], up[v]) for v in range(n)]
    best = min(heights)
    return [v for v in range(n) if heights[v] == best]


for n, e, ans in examples:
    assert sorted(find_mht_reroot(n, e)) == sorted(ans)
print("PASS ví dụ (rerooting)")

PASS ví dụ (rerooting)


**Độ phức tạp Cách 4:** Time `O(n)`; Space `O(n)`. Tổng quát hơn: cho **chiều cao của mọi gốc**, dùng được cho nhiều bài đổi gốc khác.

### Cách 5 — Bóc lá bằng hàng đợi `deque` (theo từng nút)

**Giải thích.** Cũng là Cách 2 nhưng không tách lớp tường minh: dùng `deque` các lá và theo dõi `remaining`; mỗi lượt xử lý đúng số phần tử hiện có trong hàng đợi (một "tầng"), trả về nội dung hàng đợi khi `remaining ≤ 2`. Dạng này là mẫu **Kahn theo tầng**.

In [11]:
def find_mht_peel_deque(n: int, edges: List[List[int]]) -> List[int]:
    if n == 1:
        return [0]
    adj = defaultdict(set)
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)
    q = deque(i for i in range(n) if len(adj[i]) == 1)
    remaining = n
    while remaining > 2:
        for _ in range(len(q)):                           # một tầng
            leaf = q.popleft()
            remaining -= 1
            nb = adj[leaf].pop()                           # lá chỉ có đúng một hàng xóm
            adj[nb].remove(leaf)
            if len(adj[nb]) == 1:
                q.append(nb)
    return list(q)


for n, e, ans in examples:
    assert sorted(find_mht_peel_deque(n, e)) == sorted(ans)
print("PASS ví dụ (deque)")

PASS ví dụ (deque)


**Độ phức tạp Cách 5:** Time `O(n)`; Space `O(n)`.

> Cách 2 là cách nộp bài (ngắn, tuyến tính). Cách 3 cho thấy gốc rễ lý thuyết. Cách 4 mạnh nhất khi cần chiều cao mọi gốc. Cách 1 dùng làm oracle.

## 8. TRACE từng bước

Bóc lá trên `n = 6`, `edges = [[3,0],[3,1],[3,2],[3,4],[5,4]]`. In từng lớp lá và số nút còn lại.

In [12]:
def trace_peel(n, edges):
    if n <= 2:
        print("n <= 2: mọi nút là tâm", list(range(n)))
        return list(range(n))
    adj = build_adj(n, edges)
    deg = [len(a) for a in adj]
    leaves = [i for i in range(n) if deg[i] == 1]
    remaining = n
    step = 0
    while remaining > 2:
        step += 1
        print(f"lượt {step}: còn {remaining} nút, lá = {leaves}")
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for nb in adj[leaf]:
                deg[nb] -= 1
                if deg[nb] == 1:
                    nxt.append(nb)
        leaves = nxt
    print(f"dừng: còn {remaining} nút, tâm = {leaves}")
    return leaves


assert sorted(trace_peel(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]])) == [3, 4]

lượt 1: còn 6 nút, lá = [0, 1, 2, 5]
dừng: còn 2 nút, tâm = [3, 4]


Bảng tóm tắt:

| lượt | nút còn lại | lá bị bóc | bậc mới |
|---|---|---|---|
| 1 | 6 | `0, 1, 2, 5` | `deg[3]` từ 4 xuống 1; `deg[4]` từ 2 xuống 1 |
| 2 | 2 | (dừng) | tâm là `3` và `4` |

Sau lượt 1 chỉ còn `3` và `4`, hai nút này đều đã bậc `1`: dừng vì `remaining = 2`. Đường kính là `0 − 3 − 4 − 5` (3 cạnh, lẻ), nên có hai tâm.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bóc lá theo lớp), đúng chữ ký của LeetCode.

In [13]:
class Solution:
    def findMinHeightTrees(self, n: int, edges: List[List[int]]) -> List[int]:
        if n <= 2:
            return list(range(n))
        g = [[] for _ in range(n)]
        deg = [0] * n
        for a, b in edges:
            g[a].append(b)
            g[b].append(a)
            deg[a] += 1
            deg[b] += 1
        leaves = [i for i in range(n) if deg[i] == 1]
        remaining = n
        while remaining > 2:
            remaining -= len(leaves)
            nxt = []
            for leaf in leaves:
                for nb in g[leaf]:
                    deg[nb] -= 1
                    if deg[nb] == 1:
                        nxt.append(nb)
            leaves = nxt
        return leaves


for n, e, ans in examples:
    assert sorted(Solution().findMinHeightTrees(n, e)) == sorted(ans)
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Chiều cao của gốc `r`** = khoảng cách từ `r` đến nút xa nhất `e(r)`. Nút xa nhất từ `r` luôn là một đầu của một đường kính nào đó, nên `e(r) ≥ ⌈D/2⌉` (`r` không thể gần cả hai đầu của một đường kính hơn nửa độ dài).
- **Tâm đạt dấu bằng:** nút giữa của đường kính có `e = ⌈D/2⌉` (vì mọi nút khác cách tâm một đoạn không quá một nửa, nếu không sẽ tồn tại đường dài hơn `D`).
- **Chỉ có 1 hoặc 2 tâm:** mọi đường kính đều đi qua cùng các tâm; nếu `D` chẵn, nút giữa duy nhất; nếu `D` lẻ, hai nút giữa kề nhau. Nút nào khác có `e > ⌈D/2⌉`.
- **Bóc lá đúng:** hai đầu mọi đường kính đều là lá; bỏ mọi lá làm đường kính giảm đúng 2 và tâm không đổi. Lặp lại cho đến khi còn 1 nút (`D` chẵn) hoặc 2 nút (`D` lẻ).
- **Điều kiện dừng `remaining > 2`:** khi còn đúng 2 nút kề nhau, chúng đều là lá; nếu tiếp tục bóc sẽ bỏ cả hai và mất đáp án.
- **Rerooting:** chiều cao gốc `v` = `max(down[v], up[v])`; `up` truyền từ cha xuống con kèm "nhánh anh em xa nhất": hai giá trị lớn nhất đủ để loại trừ nhánh của chính con đó.

## 11. Phân tích Time / Space Complexity

- **Cách 1 (thử mọi gốc):** Time `O(n²)`; Space `O(n)`.
- **Cách 2 (bóc lá theo lớp):** Time `O(n)`; Space `O(n)`.
- **Cách 3 (hai BFS):** Time `O(n)`; Space `O(n)`.
- **Cách 4 (rerooting):** Time `O(n)`; Space `O(n)`.
- **Cách 5 (deque):** Time `O(n)`; Space `O(n)` (dùng `set` nên hằng số lớn hơn).
- Với `n = 2·10⁴`: cách tuyến tính xử lý trong vài chục ms.

## 12. Bộ test

Bài có **nhiều đáp án đúng về thứ tự**, nên so sánh bằng **danh sách đã sắp xếp**. Ngoài ra kiểm tra thêm các tính chất: có `1` hoặc `2` nút, và chiều cao khi chọn các nút này bằng chiều cao nhỏ nhất.

In [14]:
impls = {
    "brute": find_mht_brute,
    "peel": find_mht_peel,
    "diameter": find_mht_diameter,
    "reroot": find_mht_reroot,
    "deque": find_mht_peel_deque,
    "submit": lambda n, edges: Solution().findMinHeightTrees(n, edges),
}
FAST = ("peel", "diameter", "reroot", "deque", "submit")


def check(n, edges, expected=None, names=None):
    base = sorted(find_mht_brute(n, edges)) if n <= 80 else sorted(find_mht_peel(n, edges))
    if expected is not None:
        assert base == sorted(expected), (n, edges, base, expected)
    for name, fn in impls.items():
        if names is not None and name not in names:
            continue
        if name == "brute" and n > 80:
            continue
        got = sorted(fn(n, [list(e) for e in edges]))
        assert got == base, (name, n, edges, got, base)
    return base


for n, e, ans in examples:
    check(n, e, ans)
print("PASS 2 ví dụ của đề (so sánh tập đã sắp)")

PASS 2 ví dụ của đề (so sánh tập đã sắp)


### 12.1. Các trường hợp biên

In [15]:
cases = [
    (1, [], [0], "một nút duy nhất, không có cạnh"),
    (2, [[0, 1]], [0, 1], "hai nút: cả hai đều là tâm"),
    (3, [[0, 1], [1, 2]], [1], "đường đi 3 nút"),
    (3, [[2, 0], [1, 2]], [2], "đường đi 3 nút, nhãn xáo trộn"),
    (4, [[0, 1], [1, 2], [2, 3]], [1, 2], "đường đi 4 nút: hai tâm"),
    (5, [[0, 1], [1, 2], [2, 3], [3, 4]], [2], "đường đi 5 nút"),
    (5, [[0, 1], [0, 2], [0, 3], [0, 4]], [0], "sao (star)"),
    (6, [[0, 1], [0, 2], [0, 3], [0, 4], [0, 5]], [0], "sao 6 nút"),
    (7, [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]], [0], "cây nhị phân đầy đủ 7 nút"),
    (6, [[0, 1], [1, 2], [2, 3], [3, 4], [2, 5]], [2], "chổi: tâm ở nhánh"),
]
for n, e, ans, why in cases:
    assert check(n, e, ans) == sorted(ans)
    print("PASS", why, "->", ans)

PASS một nút duy nhất, không có cạnh -> [0]
PASS hai nút: cả hai đều là tâm -> [0, 1]
PASS đường đi 3 nút -> [1]
PASS đường đi 3 nút, nhãn xáo trộn -> [2]
PASS đường đi 4 nút: hai tâm -> [1, 2]
PASS đường đi 5 nút -> [2]
PASS sao (star) -> [0]
PASS sao 6 nút -> [0]
PASS cây nhị phân đầy đủ 7 nút -> [0]
PASS chổi: tâm ở nhánh -> [2]


### 12.2. Tính chất, phản ví dụ và nhiều đáp án đúng

In [16]:
rnd = random.Random(310)

# Tính chất: đáp án có 1 hoặc 2 nút; nếu 2 thì kề nhau; mọi nút đáp án có chiều cao = nhỏ nhất; không nút nào khác đạt
for _ in range(400):
    n = rnd.randint(1, 40)
    edges = random_tree(n, rnd)
    ans = sorted(find_mht_peel(n, edges))
    adj = build_adj(n, edges)
    hs = [height_from(adj, r) for r in range(n)]
    assert len(ans) in (1, 2)
    assert all(hs[r] == min(hs) for r in ans) and sum(h == min(hs) for h in hs) == len(ans)
    if len(ans) == 2:
        assert ans[1] in adj[ans[0]]
print("PASS tính chất: 1 hoặc 2 tâm, kề nhau, đúng độ cao nhỏ nhất")

# Trả lời nhiều thứ tự đều được chấp nhận: so sánh tập
assert sorted([4, 3]) == sorted([3, 4])

# Phản ví dụ: "nút bậc cao nhất" KHÔNG phải luôn là tâm
found = None
for _ in range(2000):
    n = rnd.randint(6, 14)
    edges = random_tree(n, rnd)
    adj = build_adj(n, edges)
    top = max(range(n), key=lambda v: len(adj[v]))
    if top not in find_mht_peel(n, edges):
        found = (n, edges, top)
        break
assert found is not None
print("phản ví dụ (nút bậc cao nhất không phải tâm):", found)

PASS tính chất: 1 hoặc 2 tâm, kề nhau, đúng độ cao nhỏ nhất
phản ví dụ (nút bậc cao nhất không phải tâm): (10, [[5, 1], [1, 3], [9, 0], [8, 7], [2, 3], [9, 8], [7, 3], [6, 8], [4, 7]], 3)


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for trial in range(1200):
    n = rnd.randint(1, 30)
    check(n, random_tree(n, rnd))

for trial in range(30):                                  # n vừa: vẫn so với brute (n <= 80)
    n = rnd.randint(40, 80)
    check(n, random_tree(n, rnd))

for trial in range(20):                                  # n lớn: so các cách tuyến tính với nhau
    n = rnd.randint(1000, 20000)
    check(n, random_tree(n, rnd), names=FAST)

# Hình dạng cực đoan ở n = 20000
n = 20000
path = [[i, i + 1] for i in range(n - 1)]
assert check(n, path, [n // 2 - 1, n // 2], names=FAST) == [n // 2 - 1, n // 2]
star = [[0, i] for i in range(1, n)]
assert check(n, star, [0], names=FAST) == [0]
print("PASS 1200 test nhỏ, 30 test vừa (có brute), 20 test lớn, đường đi và sao n = 20000")

PASS 1200 test nhỏ, 30 test vừa (có brute), 20 test lớn, đường đi và sao n = 20000


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Điều kiện dừng sai (`remaining > 1`)

In [18]:
def sai_dieu_kien_dung(n, edges):
    if n <= 2:
        return list(range(n))
    adj = build_adj(n, edges)
    deg = [len(a) for a in adj]
    leaves = [i for i in range(n) if deg[i] == 1]
    remaining = n
    while remaining > 1:                                # sai: phải là > 2
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for nb in adj[leaf]:
                deg[nb] -= 1
                if deg[nb] == 1:
                    nxt.append(nb)
        leaves = nxt
    return leaves


n, e, ans = examples[1]
print("remaining > 1 ->", sai_dieu_kien_dung(n, e), "| đúng:", ans)
assert sorted(sai_dieu_kien_dung(n, e)) != sorted(ans)

remaining > 1 -> [] | đúng: [3, 4]


**Vì sao sai:** khi còn đúng hai tâm kề nhau, cả hai đều là lá. Nếu vòng lặp còn tiếp tục sẽ bóc luôn cả hai và trả về danh sách rỗng. Quy tắc: dừng khi còn `≤ 2` nút.

### Lỗi 2: Quên trường hợp `n = 1`

In [19]:
def sai_quen_n1(n, edges):
    adj = build_adj(n, edges)
    deg = [len(a) for a in adj]
    leaves = [i for i in range(n) if deg[i] == 1]      # n = 1: không có lá nào (bậc 0)
    remaining = n
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for nb in adj[leaf]:
                deg[nb] -= 1
                if deg[nb] == 1:
                    nxt.append(nb)
        leaves = nxt
    return leaves


print("n = 1 ->", sai_quen_n1(1, []), "| đúng: [0]")
assert sai_quen_n1(1, []) != [0]

n = 1 -> [] | đúng: [0]


**Vì sao sai:** nút duy nhất có bậc `0`, không bao giờ vào danh sách lá, nên kết quả rỗng. Phải xử lý riêng `n ≤ 2` (hoặc khởi tạo lá là nút bậc `≤ 1`).

### Lỗi 3: Chỉ trả về một tâm khi đường kính lẻ

In [20]:
def sai_mot_tam(n, edges):
    if n == 1:
        return [0]
    adj = build_adj(n, edges)
    u, _, _ = bfs_far(adj, 0)
    v, dist, par = bfs_far(adj, u)
    path = [v]
    while par[path[-1]] != -1:
        path.append(par[path[-1]])
    return [path[len(path) // 2]]                       # sai: bỏ mất tâm thứ hai khi D lẻ


n, e, ans = examples[1]
print("một tâm ->", sai_mot_tam(n, e), "| đúng:", ans)
assert sorted(sai_mot_tam(n, e)) != sorted(ans)

một tâm -> [4] | đúng: [3, 4]


**Vì sao sai:** đường kính `0 − 3 − 4 − 5` có 3 cạnh (lẻ), nên cả `3` và `4` cho chiều cao 2. Phải trả cả hai.

### Lỗi 4: Chọn nút bậc cao nhất làm gốc

In [21]:
n0, e0, top0 = found
print("cây phản ví dụ:", n0, e0)
print("nút bậc cao nhất =", top0, "| tâm thật =", sorted(find_mht_peel(n0, e0)))
assert top0 not in find_mht_peel(n0, e0)

cây phản ví dụ: 10 [[5, 1], [1, 3], [9, 0], [8, 7], [2, 3], [9, 8], [7, 3], [6, 8], [4, 7]]
nút bậc cao nhất = 3 | tâm thật = [7]


**Vì sao sai:** bậc đo "bao nhiêu nhánh", còn chiều cao đo "nhánh dài nhất". Một nút có nhiều nhánh ngắn vẫn có thể lệch khỏi giữa đường kính. Tâm phải xác định từ **độ dài**, không từ bậc.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Chọn gốc để chiều cao/khoảng cách xa nhất nhỏ nhất" → **tâm của đường kính**; đáp án luôn 1 hoặc 2 nút.
- Hai cách làm tuyến tính chuẩn: **bóc lá** (Kahn trên lá, dừng ở `≤ 2` nút) hoặc **hai lượt BFS** (đường kính, lấy giữa).
- `n ≤ 2` trả thẳng `range(n)`: tránh mọi lỗi biên của `n = 1`.
- Cần chiều cao **mọi gốc** → **rerooting** (`down` và `up`, giữ hai giá trị lớn nhất của các con).
- Bóc lá theo lớp cũng là mẫu cho "đồ thị vô hướng + bậc" (ví dụ phát hiện chu trình bằng cách bóc lá còn dư).
- Test bài nhiều đáp án: so sánh **tập đã sắp** và kiểm tra **tính chất** thay vì thứ tự.

## 15. Biến thể của bài

1. **Chiều cao nhỏ nhất** (chỉ giá trị `⌈D/2⌉`).
2. **Độ dài đường kính của cây** (LeetCode 1245 Tree Diameter, Premium).
3. **Tổng khoảng cách từ mỗi nút đến mọi nút khác** (LeetCode 834): rerooting.
4. **Chiều cao của mọi gốc** (dùng cách 4).
5. **Cây có trọng số cạnh:** tâm theo tổng trọng số (cần dung sai khi so sánh hai tâm).

In [22]:
def mht_height(n, edges):
    """Biến thể 1: chiều cao nhỏ nhất."""
    if n == 1:
        return 0
    adj = build_adj(n, edges)
    u, _, _ = bfs_far(adj, 0)
    v, dist, _ = bfs_far(adj, u)
    return (dist[v] + 1) // 2


def tree_diameter(n, edges):
    """Biến thể 2: số cạnh của đường kính."""
    if n == 1:
        return 0
    adj = build_adj(n, edges)
    u, _, _ = bfs_far(adj, 0)
    v, dist, _ = bfs_far(adj, u)
    return dist[v]


def sum_distances(n, edges):
    """Biến thể 3 (834): tổng khoảng cách từ mỗi nút tới mọi nút, bằng rerooting."""
    adj = build_adj(n, edges)
    parent = [-1] * n
    order = [0]
    for u in order:
        for v in adj[u]:
            if v != parent[u]:
                parent[v] = u
                order.append(v)
    size = [1] * n
    down = [0] * n
    for u in reversed(order):
        p = parent[u]
        if p >= 0:
            size[p] += size[u]
            down[p] += down[u] + size[u]
    ans = [0] * n
    ans[0] = down[0]
    for u in order[1:]:
        ans[u] = ans[parent[u]] - size[u] + (n - size[u])
    return ans


def all_heights(n, edges):
    """Biến thể 4: chiều cao khi chọn từng nút làm gốc."""
    adj = build_adj(n, edges)
    return [height_from(adj, r) for r in range(n)]


rnd = random.Random(15)
for _ in range(200):
    n = rnd.randint(1, 25)
    edges = random_tree(n, rnd)
    hs = all_heights(n, edges)
    assert mht_height(n, edges) == min(hs)
    assert tree_diameter(n, edges) == max(hs)
    adj = build_adj(n, edges)
    brute_sum = []
    for r in range(n):
        dist = [-1] * n
        dist[r] = 0
        q = deque([r])
        while q:
            x = q.popleft()
            for y in adj[x]:
                if dist[y] < 0:
                    dist[y] = dist[x] + 1
                    q.append(y)
        brute_sum.append(sum(dist))
    assert sum_distances(n, edges) == brute_sum
assert mht_height(4, [[1, 0], [1, 2], [1, 3]]) == 1
print("PASS biến thể: chiều cao nhỏ nhất, đường kính, tổng khoảng cách, chiều cao mọi gốc")

PASS biến thể: chiều cao nhỏ nhất, đường kính, tổng khoảng cách, chiều cao mọi gốc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi gốc | BFS từ từng nút | `O(n²)` | `O(n)` | oracle |
| 2. Bóc lá theo lớp | Kahn trên lá, dừng ở `≤ 2` nút | `O(n)` | `O(n)` | **cách nộp bài** |
| 3. Hai BFS | đường kính, lấy giữa | `O(n)` | `O(n)` | giải thích lý thuyết |
| 4. Rerooting | `down`/`up` cho mọi gốc | `O(n)` | `O(n)` | cần chiều cao mọi gốc |
| 5. Bóc lá bằng deque | tầng bằng `len(q)` | `O(n)` | `O(n)` | gọn, dùng `set` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `find_mht_peel(4, [[0,1],[1,2],[2,3]])` và `find_mht_peel(1, [])`.

**Bài 2 (tính tay).** Với đường đi `0 − 1 − 2 − 3 − 4` tính chiều cao khi chọn từng nút làm gốc, rồi cho biết MHT.

**Bài 3 (điền TODO).** Hoàn thành `find_mht_todo` theo Cách 2 (bóc lá).

In [23]:
def find_mht_todo(n, edges):
    # TODO 1: nếu n <= 2 trả list(range(n))
    # TODO 2: dựng danh sách kề g và mảng bậc deg
    # TODO 3: leaves = các nút có deg == 1; remaining = n
    # TODO 4: while remaining > 2: trừ len(leaves), giảm bậc hàng xóm, nút mới có bậc 1 vào lớp kế
    # TODO 5: trả leaves
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_mht_diameter2(n, edges)` bằng **DFS lặp** (dùng ngăn xếp) thay vì BFS để tìm đường kính, rồi lấy tâm.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_mht(n, edges)` trả về **số** MHT (chỉ `1` hoặc `2`) mà không cần liệt kê. (b) Viết `diameter_weighted(n, wedges)` cho cây có trọng số cạnh dương `[a, b, w]` bằng hai lượt DFS lặp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: đường đi 4 nút có hai tâm; `n = 1` là trường hợp biên. Bài 2: chiều cao gốc = khoảng cách tới đầu xa nhất.
- **Mức 2 (vừa):** Bài 1: `[1, 2]` và `[0]`. Bài 2: chiều cao `[4, 3, 2, 3, 4]`, MHT là `[2]`. Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: DFS lặp bằng ngăn xếp `(nút, cha)`; ghi `dist` và `parent`; lấy nút `dist` lớn nhất. Bài 5(a): hai tâm khi và chỉ khi đường kính có số cạnh lẻ, tức số tâm là `1 + D % 2`. Bài 5(b): hai lượt duyệt, cộng trọng số cạnh khi đi xuống, lấy nút có tổng lớn nhất.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert sorted(find_mht_peel(4, [[0, 1], [1, 2], [2, 3]])) == [1, 2]
assert find_mht_peel(1, []) == [0]

# Bài 2
assert all_heights(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == [4, 3, 2, 3, 4]

# Bài 3
def find_mht_todo_ans(n, edges):
    if n <= 2:
        return list(range(n))
    g = [[] for _ in range(n)]
    deg = [0] * n
    for a, b in edges:
        g[a].append(b)
        g[b].append(a)
        deg[a] += 1
        deg[b] += 1
    leaves = [i for i in range(n) if deg[i] == 1]
    remaining = n
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            for nb in g[leaf]:
                deg[nb] -= 1
                if deg[nb] == 1:
                    nxt.append(nb)
        leaves = nxt
    return leaves


# Bài 4
def farthest_iter(adj, src):
    n = len(adj)
    dist = [-1] * n
    par = [-1] * n
    dist[src] = 0
    stack = [src]
    best = src
    while stack:
        u = stack.pop()
        if dist[u] > dist[best]:
            best = u
        for v in adj[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1
                par[v] = u
                stack.append(v)
    return best, dist, par


def find_mht_diameter2(n, edges):
    if n == 1:
        return [0]
    adj = build_adj(n, edges)
    u, _, _ = farthest_iter(adj, 0)
    v, dist, par = farthest_iter(adj, u)
    path = [v]
    while par[path[-1]] != -1:
        path.append(par[path[-1]])
    D = len(path) - 1
    return [path[D // 2]] if D % 2 == 0 else [path[D // 2], path[D // 2 + 1]]


# Bài 5
def count_mht(n, edges):
    return 1 if n == 1 else 1 + tree_diameter(n, edges) % 2


def diameter_weighted(n, wedges):
    adj = [[] for _ in range(n)]
    for a, b, w in wedges:
        adj[a].append((b, w))
        adj[b].append((a, w))

    def far(src):
        dist = [-1] * n
        dist[src] = 0
        stack = [src]
        best = src
        while stack:
            u = stack.pop()
            if dist[u] > dist[best]:
                best = u
            for v, w in adj[u]:
                if dist[v] < 0:
                    dist[v] = dist[u] + w
                    stack.append(v)
        return best, dist

    u, _ = far(0)
    v, dist = far(u)
    return dist[v]


rnd = random.Random(19)
for _ in range(300):
    n = rnd.randint(1, 30)
    edges = random_tree(n, rnd)
    base = sorted(find_mht_brute(n, edges))
    assert sorted(find_mht_todo_ans(n, edges)) == base
    assert sorted(find_mht_diameter2(n, edges)) == base
    assert count_mht(n, edges) == len(base)
    wedges = [[a, b, rnd.randint(1, 9)] for a, b in edges]
    adjw = [[] for _ in range(n)]
    for a, b, w in wedges:
        adjw[a].append((b, w))
        adjw[b].append((a, w))
    best = 0
    for r in range(n):                                  # brute: khoảng cách xa nhất giữa mọi cặp
        dist = {r: 0}
        stack = [r]
        while stack:
            x = stack.pop()
            for y, w in adjw[x]:
                if y not in dist:
                    dist[y] = dist[x] + w
                    stack.append(y)
        best = max(best, max(dist.values()))
    assert diameter_weighted(n, wedges) == best
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bóc lá theo lớp)

```python
class Solution:
    def findMinHeightTrees(self, n: int, edges: List[List[int]]) -> List[int]:
        if n <= 2:
            return list(range(n))
        g = [[] for _ in range(n)]
        deg = [0] * n
        for a, b in edges:
            g[a].append(b); g[b].append(a)
            deg[a] += 1; deg[b] += 1
        leaves = [i for i in range(n) if deg[i] == 1]
        remaining = n
        while remaining > 2:
            remaining -= len(leaves)
            nxt = []
            for leaf in leaves:
                for nb in g[leaf]:
                    deg[nb] -= 1
                    if deg[nb] == 1:
                        nxt.append(nb)
            leaves = nxt
        return leaves
```

**Checklist**

- [ ] Đã xử lý `n = 1` và `n = 2` riêng (`range(n)`)?
- [ ] Điều kiện dừng là `remaining > 2` (không phải `> 1` hay `> 0`)?
- [ ] Nút vào lớp kế khi bậc **đúng bằng 1** sau khi trừ?
- [ ] Trả về **cả hai** tâm khi đường kính có số cạnh lẻ?
- [ ] Kiểm thử so sánh **tập đã sắp xếp**, không so thứ tự?
- [ ] Không dựa vào "bậc cao nhất" để chọn tâm?
- [ ] Đã đối chiếu với brute force (BFS từ mọi gốc) trên cây ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `findMinHeightTrees(self, n: int, edges: List[List[int]]) -> List[int]`?